In [ ]:
# Cell: setup
import os
import json
from pathlib import Path
from datetime import datetime
import pandas as pd

RAW = Path("data/raw/octus")
OUT = Path("data/processed/simfin")
OUT.mkdir(parents=True, exist_ok=True)

companies = json.loads((RAW / "company_metadata.json").read_text(encoding="utf-8"))
octus_companies_df = pd.DataFrame(companies)[["octus_company_id", "company_name", "sub_industry"]].copy()
octus_companies_df

In [ ]:
# Cell: load SimFin companies via simfin Python package (preferred for simplicity)
# pip install simfin rapidfuzz

import simfin as sf
from rapidfuzz import fuzz, process

SIMFIN_API_KEY = os.getenv("SIMFIN_API_KEY")
SIMFIN_DATA_DIR = os.getenv("SIMFIN_DATA_DIR", "data/cache/simfin_py")

assert SIMFIN_API_KEY, "Set SIMFIN_API_KEY in your environment"

sf.set_api_key(SIMFIN_API_KEY)
sf.set_data_dir(SIMFIN_DATA_DIR)

# Adjust market as needed; keep it configurable in the main app settings.
simfin_companies_df = sf.load_companies(market="us")

simfin_companies_df.head(), simfin_companies_df.shape

In [ ]:
# Cell: fuzzy match Octus company_name -> SimFin company name/ticker
# SimFin companies df is typically indexed by ticker; inspect columns to pick name fields.
print("SimFin columns:", list(simfin_companies_df.columns))

# Choose a best-effort name column (fallbacks).
name_col = None
for c in ["Company Name", "Name", "company_name", "companyName"]:
    if c in simfin_companies_df.columns:
        name_col = c
        break

# If SimFin companies DataFrame uses a different schema, inspect and set manually.
assert name_col is not None, "Could not find a company name column; inspect simfin_companies_df.columns"

choices = simfin_companies_df[name_col].astype(str).tolist()

rows = []
tickers = simfin_companies_df.index.astype(str).tolist()

for _, r in octus_companies_df.iterrows():
    q = str(r["company_name"])
    match = process.extractOne(q, choices, scorer=fuzz.WRatio)
    if match:
        matched_name, score, idx = match
        ticker = tickers[idx]
        rows.append({
            "octus_company_id": r["octus_company_id"],
            "company_name": r["company_name"],
            "sub_industry": r["sub_industry"],
            "suggested_simfin_name": matched_name,
            "suggested_ticker": ticker,
            "suggested_simfin_id": None,  # optional: fill if available in your SimFin company schema
            "match_score": float(score),
            "status": "auto_matched" if score >= 90 else "needs_review",
            "updated_at": datetime.utcnow().isoformat()
        })

mapping_df = pd.DataFrame(rows).sort_values(["status", "match_score"], ascending=[True, False])
mapping_df

In [ ]:
# Cell: inspect matches needing review and manually confirm/reject
needs_review = mapping_df[mapping_df["status"] == "needs_review"]
print(f"Mappings needing review: {len(needs_review)}")
needs_review[["company_name", "suggested_ticker", "suggested_simfin_name", "match_score"]]

In [ ]:
# Cell: promote auto_matched to confirmed (matches setting auto_promote_matched=True)
# Manual overrides can be applied here before saving.

# Example: manually reject a bad match
# mapping_df.loc[mapping_df["octus_company_id"] == "XXXX", "status"] = "rejected"

# Example: manually confirm a needs_review entry
# mapping_df.loc[mapping_df["octus_company_id"] == "YYYY", "status"] = "confirmed"

# Promote all auto_matched to confirmed (mirrors auto_promote_matched=True setting)
mapping_df.loc[mapping_df["status"] == "auto_matched", "status"] = "confirmed"

print("Status distribution after promotion:")
mapping_df["status"].value_counts()

In [ ]:
# Cell: save mapping to disk
out_path = OUT / "company_map.csv"
mapping_df.to_csv(out_path, index=False)
print(f"Mapping saved to {out_path}")
mapping_df[["octus_company_id", "company_name", "suggested_ticker", "match_score", "status"]]